# Fine-tuning BETO para análisis de sentimiento en español

In [ ]:
!pip install transformers datasets evaluate seqeval scikit-learn -q

## Importar librerías y configuración

In [ ]:
import torch
import pandas as pd
import evaluate
import numpy as np

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification,
    pipeline
)

from sklearn.model_selection import train_test_split

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

metric = evaluate.load("seqeval")

label_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

Device: cuda:0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## Tokenización

In [ ]:
def tokenize_and_labels(examples, tokenizer):
    label_all_tokens = True

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(i)
        current_word = None
        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != current_word:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)

            current_word = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

## Métricas

In [ ]:
def compute_metrics(eval_pred):

    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=-1)

    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_names[l] for l in label if l != -100]
        for label in labels
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

## Cargar dataset y tokenizer

In [ ]:
transformer_model = "google-bert/bert-base-uncased"

label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for i, label in enumerate(label_names)}

dataset = load_dataset("lhoestq/conll2003")

train_dataset = dataset["train"]
eval_dataset = dataset["validation"]

tokenizer = AutoTokenizer.from_pretrained(transformer_model)

train_dataset = train_dataset.map(
    tokenize_and_labels,
    batched=True,
    fn_kwargs={"tokenizer": tokenizer}
)

eval_dataset = eval_dataset.map(
    tokenize_and_labels,
    batched=True,
    fn_kwargs={"tokenizer": tokenizer}
)

print(train_dataset[0])

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/281k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0], 'input_ids': [101, 7327, 19164, 2446, 2655, 2000, 17757, 2329, 12559, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, -100]}


In [ ]:
def model_init():
    return AutoModelForTokenClassification.from_pretrained(
        transformer_model,
        id2label=id2label,
        label2id=label2id
    )

data_collator = DataCollatorForTokenClassification(tokenizer)

## Argumentos de entrenamiento

In [ ]:
training_args = TrainingArguments(

    output_dir="./log",

    num_train_epochs=6,

    eval_strategy="epoch",
    save_strategy="epoch",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    fp16=True, # Para entrenar más rápido
    save_total_limit=1,

    learning_rate=2e-5,
    weight_decay=0.01,

    seed=1
)

In [ ]:
trainer = Trainer(

    model_init=model_init,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

trainer.train()
trainer.save_model("./model_finetuned")
tokenizer.save_pretrained("./model_finetuned")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly in

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly in

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./model_finetuned/tokenizer_config.json', './model_finetuned/tokenizer.json')

In [ ]:
# Inferencia

classifier = pipeline(
    "ner",
    model="./model_finetuned",
    device=0 if torch.cuda.is_available() else -1
)

text = "Hello Jack, today is February 4th in Spain, Apple company"

classifier(text)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[{'entity': 'B-MISC',
  'score': np.float32(0.69593483),
  'index': 1,
  'word': 'hello',
  'start': 0,
  'end': 5},
 {'entity': 'I-MISC',
  'score': np.float32(0.67510855),
  'index': 2,
  'word': 'jack',
  'start': 6,
  'end': 10},
 {'entity': 'B-LOC',
  'score': np.float32(0.99948823),
  'index': 9,
  'word': 'spain',
  'start': 37,
  'end': 42},
 {'entity': 'B-ORG',
  'score': np.float32(0.99819785),
  'index': 11,
  'word': 'apple',
  'start': 44,
  'end': 49},
 {'entity': 'I-ORG',
  'score': np.float32(0.9893106),
  'index': 12,
  'word': 'company',
  'start': 50,
  'end': 57}]